# 01 — Data Quality Audit

This notebook performs a comprehensive data quality audit of the Indian startup funding dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
from src.data_loader import load_raw_csv, run_data_audit, validate_schema
from src.preprocessing import preprocess, save_processed
from src.config import RAW_CSV, CLEANED_CSV, QUALITY_REPORT


## 1. Load Raw Data

In [ ]:
df = load_raw_csv()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Schema Validation

In [ ]:
validate_schema(df)
print("Schema validation passed.")

## 3. Data Quality Report

In [ ]:
df_normalized, report = run_data_audit()
print(json.dumps(report, indent=2, default=str))

## 4. Missing Value Analysis

In [ ]:
missing = df_normalized.isnull().sum()
missing_pct = (missing / len(df_normalized) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

## 5. Duplicate Analysis

In [ ]:
exact_duplicates = df_normalized.duplicated(keep=False).sum()
print(f"Rows involved in exact duplication: {exact_duplicates}")
print(f"Unique rows after deduplication: {len(df_normalized.drop_duplicates())}")

## 6. Date Parsing

In [ ]:
from src.data_loader import parse_date
df_normalized['date_parsed'] = df_normalized['date_dd_mm_yyyy'].apply(parse_date)
valid_dates = df_normalized['date_parsed'].dropna()
print(f"Valid dates: {len(valid_dates)}")
print(f"Invalid dates: {df_normalized['date_parsed'].isna().sum()}")
print(f"Date range: {valid_dates.min()} to {valid_dates.max()}")

## 7. Amount Parsing

In [ ]:
from src.data_loader import parse_amount
df_normalized['amount_parsed'] = df_normalized['amount_in_usd'].apply(parse_amount)
valid_amounts = df_normalized['amount_parsed'].dropna()
print(f"Valid amounts: {len(valid_amounts)}")
print(f"Missing/invalid amounts: {df_normalized['amount_parsed'].isna().sum()}")
print(f"Amount range: ${valid_amounts.min():,.0f} to ${valid_amounts.max():,.0f}")
print(f"Median amount: ${valid_amounts.median():,.0f}")

## 8. Run Preprocessing Pipeline

In [ ]:
df_clean = preprocess(df)
save_processed(df_clean)
print(f"Cleaned data saved: {len(df_clean)} rows")
df_clean.head()

## 9. Summary

Key findings from the data quality audit:

- The dataset contains funding events from January 2015 to January 2020
- Missing values exist in amount, date, and categorical fields
- Some duplicate rows exist (exact duplicates)
- Funding amounts use Indian-style comma separators
- City names have variants that need normalization
- All amounts are in USD (consistent units)